# Wall-X / WALL-OSS resident adapter (`--backend wallx`)

Serves Wall-X through lerobot's own `WallXPolicy` from a resident Flask adapter, `control/vla_bridge/wallx_server.py`, through an ngrok HTTP tunnel.

| | |
|---|---|
| Lane | HTTP adapter (kernel Python, `lerobot[wallx]==0.6.1`) |
| Port → tunnel | 3002 → ngrok HTTP |
| GPU runtime | A100 High-RAM |
| Gated HF licences | none |
| Sign-offs before running | none |
| Default checkpoint | `Zekai-Chen/wallx_pick_place_3cam_200ep@ef8ce55f9772` |
| Server log | `/content/wallx_server.log` |
| Embedded files (written to `/content`) | `colab_server_utils.py`, `wallx_server.py` |

Run top to bottom on a fresh runtime. Works from the VS Code Colab extension: nothing to upload.
Every file this notebook runs is written into `/content` by a cell below and checked against
its sha256. Run the teardown cell at the end of every session. The index of all notebooks,
with the local smoke and episode commands, is `control/vla_bridge/notebooks/README.md`.

Generated by `scripts/build_colab_notebooks.py` from `scripts/colab_notebook_specs.py` +
`control/vla_bridge` sources, so edit those and regenerate, not this file.

In [ ]:
import os

# Test 7 UAT crash investigation (Phase 12, 2026-09-27): Colab sessions that
# load multiple ML libraries (torch, opencv-headless, Fast-FoundationStereo's
# own deps) commonly end up with more than one bundled OpenMP runtime
# (libgomp vs libiomp) loaded into the same process -- the classic
# "OMP: Error #15" failure mode, which manifests as an abrupt, tracebackless
# kernel death the first time a *new* library's compute path runs for the
# first time (matches "crashes right as inference starts"). Set before any
# heavy imports so it applies process-wide to every cell below.
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")
os.environ.setdefault("OMP_NUM_THREADS", "1")
print("OpenMP duplicate-runtime guards set.")

**pyngrok legitimacy gate:** the next cell installs `pyngrok`, flagged
`SUS` in `11-RESEARCH.md`'s Package Legitimacy Audit. Do **not** run that
install cell until a human has completed the verification in this plan's
Task 3 `checkpoint:human-verify` (visit pypi.org/project/pyngrok, confirm
the `ngrok`/`alexdlaird` maintainer namespace and that `8.1.2` is a real
published version).

In [ ]:
# Colab notebook cell — DO NOT RUN until the human has completed the
# pyngrok legitimacy check in this plan's Task 3 checkpoint:human-verify
!pip install pyngrok==8.1.2

In [ ]:
# One-time token setup: HF Hub (for authenticated, faster/rate-limit-free checkpoint
# downloads) + ngrok (for the TCP tunnel). Checks for saved token files on Google Drive
# first (same MyDrive/SoARM-Research/.hf_token / .ngrok_token convention as other
# cells in this project), else falls back to a one-time getpass() prompt so tokens
# are never echoed or left in cell output/notebook history.
import getpass
from pathlib import Path

_hf_token = None
_ngrok_token = None
try:
    from google.colab import drive
    drive.mount("/content/drive")
    _hf_token_path = Path("/content/drive/MyDrive/SoARM-Research/.hf_token")
    _ngrok_token_path = Path("/content/drive/MyDrive/SoARM-Research/.ngrok_token")
    if _hf_token_path.exists():
        _hf_token = _hf_token_path.read_text().strip()
        print(f"HF token loaded from {_hf_token_path}")
    if _ngrok_token_path.exists():
        _ngrok_token = _ngrok_token_path.read_text().strip()
        print(f"ngrok token loaded from {_ngrok_token_path}")
except Exception as _exc:
    print(f"Drive mount/token read skipped ({type(_exc).__name__}: {_exc})")

if not _hf_token:
    _hf_token = getpass.getpass("HF Hub token (paste here): ")
if not _ngrok_token:
    _ngrok_token = getpass.getpass("ngrok authtoken (paste here): ")

from huggingface_hub import login
login(token=_hf_token)
print("Logged into HF Hub.")

from pyngrok import ngrok
ngrok.set_auth_token(_ngrok_token)
print("ngrok authtoken set.")

del _hf_token, _ngrok_token

## Server helper

The next cell writes `control/vla_bridge/colab_server_utils.py` into `/content/colab_server_utils.py`; the cell after it checks its sha256. The launch, diagnostic and teardown cells import it to stop stale servers, prove this notebook's server owns its port, and report logs, processes and GPU memory.

In [ ]:
%%writefile /content/colab_server_utils.py
"""Colab-side process/port helper for the per-backend policy-server notebooks.

Every notebook in control/vla_bridge/notebooks/ writes this file to
/content/colab_server_utils.py with a %%writefile cell (byte-identical to this file, checked
by a sha256 cell and by control/test_colab_notebooks.py) and imports it with
sys.path.insert(0, '/content'). Local control code never imports it.

What it guarantees for a launch cell:
- stop_servers() clears any earlier server: every process whose cmdline contains a known
  server marker, plus whatever is listening on the notebook's port.
- start_server() starts the server unbuffered (PYTHONUNBUFFERED=1) with its log in a file.
- wait_until_ready() only succeeds when the process this cell started (or one of its
  children) owns the port. A stale server that answers on the port is an error, not a
  success. It also fails on an early exit and, for patched launchers, on a missing banner.

The diagnostic helpers (log_report, process_report, gpu_report, health_report) print what a
partner needs to debug a run, and stop_server() is the teardown.

Imports: stdlib + psutil only (psutil is preinstalled on Colab).
"""

import json
import os
import re
import socket
import subprocess
import time
import urllib.request
from pathlib import Path
from typing import Callable, Mapping, Optional, Sequence

import psutil

SERVER_MARKERS = (
    "lerobot.async_inference.policy_server",
    "diffusion_policy_server.py",
    "groot_n17_server.py",
    "fastfs_server.py",
    "openvla_oft_server.py",
    "openvla_server.py",
    "octo_server.py",
    "wallx_server.py",
    "rdt1b_server.py",
    "groot_n15_server.py",
)

# Case-sensitive on purpose: "Error" covers KeyError, NotImplementedError, GatedRepoError and
# "Error in StreamActions"; lowercase prose such as "error budget" is not a match.
ERROR_PATTERN = re.compile(
    r"ERROR|Error|Traceback|Exception|OutOfMemory|out of memory|Killed|beginAllocateToPool"
)

_ACCESS_ERRORS = (psutil.AccessDenied, psutil.NoSuchProcess, psutil.ZombieProcess)


def _cmdline(proc) -> str:
    try:
        return " ".join(proc.cmdline())
    except _ACCESS_ERRORS:
        return ""
    except OSError:
        return ""


def _describe(pid: int) -> str:
    try:
        return f"PID {pid} ({_cmdline(psutil.Process(pid)) or '?'})"
    except _ACCESS_ERRORS:
        return f"PID {pid} (gone)"


def _tcp_connections(proc):
    # Colab's psutil may predate 6.0, where net_connections() replaced connections().
    method = getattr(proc, "net_connections", None) or proc.connections
    return method(kind="tcp")


def _read_lines(log_path) -> list:
    path = Path(log_path)
    if not path.exists():
        return []
    return path.read_text(errors="replace").splitlines()


def _tail(log_path, lines: int = 40) -> str:
    return "\n".join(_read_lines(log_path)[-lines:])


def listening_pids(port: int) -> set:
    """PIDs of every visible process with a TCP socket in LISTEN state on `port`."""
    pids = set()
    for proc in psutil.process_iter():
        try:
            for conn in _tcp_connections(proc):
                if conn.status == psutil.CONN_LISTEN and conn.laddr and conn.laddr.port == port:
                    pids.add(proc.pid)
        except _ACCESS_ERRORS:
            continue
        except OSError:
            continue
    return pids


def stop_servers(port: int, patterns: Sequence[str] = (), timeout_s: float = 15.0) -> list:
    """Stop every process matching a pattern plus the port's listener; return what was stopped."""
    me = os.getpid()
    targets = {}
    for proc in psutil.process_iter():
        if proc.pid == me:
            continue
        cmdline = _cmdline(proc)
        if cmdline and any(pattern and pattern in cmdline for pattern in patterns):
            targets[proc.pid] = proc
    for pid in listening_pids(port):
        if pid != me and pid not in targets:
            try:
                targets[pid] = psutil.Process(pid)
            except _ACCESS_ERRORS:
                continue
    stopped = []
    for pid, proc in sorted(targets.items()):
        description = f"PID {pid} ({_cmdline(proc) or '?'})"
        try:
            proc.terminate()
            stopped.append(description)
        except psutil.NoSuchProcess:
            continue
        except psutil.AccessDenied:
            print(f"Cannot stop {description}: access denied")
    _, alive = psutil.wait_procs(list(targets.values()), timeout=5)
    for proc in alive:
        try:
            proc.kill()
        except psutil.NoSuchProcess:
            pass
    psutil.wait_procs(alive, timeout=5)
    for line in stopped:
        print(f"Stopped {line}")
    deadline = time.monotonic() + timeout_s
    while True:
        holders = listening_pids(port) - {me}
        if not holders:
            return stopped
        if time.monotonic() >= deadline:
            raise RuntimeError(
                f"port {port} is still held by "
                + ", ".join(_describe(pid) for pid in sorted(holders))
                + f" after {timeout_s:.0f} s; stop it by hand before launching"
            )
        time.sleep(0.2)


def start_server(argv: Sequence[str], log_path,
                 env_overrides: Optional[Mapping[str, str]] = None,
                 cwd=None) -> subprocess.Popen:
    """Start `argv` unbuffered with stdout+stderr in `log_path` (truncated)."""
    env = os.environ.copy()
    env.update(env_overrides or {})
    env["PYTHONUNBUFFERED"] = "1"
    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    log = open(log_path, "wb", buffering=0)
    try:
        proc = subprocess.Popen(list(argv), stdout=log, stderr=subprocess.STDOUT, env=env,
                                cwd=None if cwd is None else str(cwd))
    finally:
        log.close()
    print(f"Started PID {proc.pid}: {' '.join(argv)}")
    print(f"Log → {log_path}")
    return proc


def _probe(port: int, health_url: Optional[str], ready: Optional[Callable[[dict], bool]]):
    """Return (answered, health): answered is True once the server counts as ready."""
    if health_url:
        try:
            with urllib.request.urlopen(health_url, timeout=2) as response:
                health = json.loads(response.read().decode())
        except Exception:
            return False, None
        if not isinstance(health, dict):
            return False, None
        ok = ready(health) if ready is not None else health.get("status") == "ready"
        return bool(ok), health
    try:
        with socket.create_connection(("127.0.0.1", port), timeout=1):
            return True, None
    except OSError:
        return False, None


def _own_pids(proc) -> set:
    pids = {proc.pid}
    try:
        pids.update(child.pid for child in psutil.Process(proc.pid).children(recursive=True))
    except _ACCESS_ERRORS:
        pass
    return pids


def wait_until_ready(proc, port: int, log_path, timeout_s: float,
                     health_url: Optional[str] = None,
                     ready: Optional[Callable[[dict], bool]] = None,
                     banner: Optional[str] = None,
                     settle_s: float = 2.0, poll_s: float = 1.0):
    """Wait until the server this cell started owns `port` and is ready.

    Returns the /health dict in health mode, None in TCP mode. Raises RuntimeError on an
    early exit, a foreign/stale owner or a missing launcher banner, and TimeoutError (after
    terminating `proc`) when the deadline passes.
    """
    deadline = time.monotonic() + timeout_s
    while time.monotonic() < deadline:
        code = proc.poll()
        if code is not None:
            raise RuntimeError(
                f"server PID {proc.pid} exited with code {code} before it was ready. "
                f"Last log lines of {log_path}:\n{_tail(log_path)}"
            )
        answered, health = _probe(port, health_url, ready)
        if answered:
            owners = listening_pids(port)
            mine = _own_pids(proc)
            if not owners or not owners <= mine:
                foreign = ", ".join(_describe(pid) for pid in sorted(owners - mine))
                foreign = foreign or "no visible process"
                raise RuntimeError(
                    f"port {port} answered but is owned by {foreign}, not the server this cell "
                    f"started (PID {proc.pid}). That is a stale server; re-run the launch cell, "
                    "which stops old servers first."
                )
            time.sleep(settle_s)
            code = proc.poll()
            if code is not None:
                raise RuntimeError(
                    f"server PID {proc.pid} exited with code {code} right after binding port "
                    f"{port}. Last log lines of {log_path}:\n{_tail(log_path)}"
                )
            if banner is not None:
                text = Path(log_path).read_text(errors="replace") if Path(log_path).exists() else ""
                if banner not in text:
                    raise RuntimeError(
                        f"expected launcher banner '{banner}' not found in {log_path}, so this "
                        "is not the patched launcher. Re-run the embed cells, then the launch cell."
                    )
            print(f"READY: PID {proc.pid} ({_cmdline(psutil.Process(proc.pid))}) owns port "
                  f"{port}; log → {log_path}")
            return health
        time.sleep(poll_s)
    if proc.poll() is None:
        proc.terminate()
        try:
            proc.wait(timeout=10)
        except subprocess.TimeoutExpired:
            proc.kill()
    raise TimeoutError(
        f"server PID {proc.pid} was not ready on port {port} within {timeout_s:.0f} s "
        f"(terminated). Last log lines of {log_path}:\n{_tail(log_path)}"
    )


def log_report(log_path, tail: int = 80, max_matches: int = 40) -> None:
    """Print the last `tail` log lines and up to `max_matches` numbered error lines."""
    path = Path(log_path)
    if not path.exists():
        print(f"{path} does not exist (has the launch cell run on this runtime?)")
        return
    lines = _read_lines(path)
    print(f"--- last {tail} lines of {path} ({len(lines)} lines total) ---")
    for line in lines[-tail:]:
        print(line)
    matches = [(number, line) for number, line in enumerate(lines, start=1)
               if ERROR_PATTERN.search(line)]
    print(f"--- {len(matches)} lines matching ERROR_PATTERN (showing up to {max_matches}) ---")
    for number, line in matches[:max_matches]:
        print(f"{number}: {line}")
    if not matches:
        print("(none)")


def process_report(port: int, expected_marker: str, proc=None) -> None:
    """Print every known server process, then check the right server owns `port`."""
    print("--- running server processes ---")
    found = False
    now = time.time()
    for candidate in psutil.process_iter():
        cmdline = _cmdline(candidate)
        if not cmdline or not any(marker in cmdline for marker in SERVER_MARKERS):
            continue
        try:
            elapsed = now - candidate.create_time()
            rss_mb = candidate.memory_info().rss / 1e6
        except _ACCESS_ERRORS:
            continue
        found = True
        print(f"PID {candidate.pid}  up {elapsed:7.0f} s  RSS {rss_mb:8.1f} MB  {cmdline}")
    if not found:
        print("(no process with a known server marker)")
    owners = listening_pids(port)
    if not owners:
        raise RuntimeError(f"nothing is listening on port {port}; run the launch cell")
    for pid in sorted(owners):
        try:
            cmdline = _cmdline(psutil.Process(pid))
        except _ACCESS_ERRORS:
            cmdline = ""
        if expected_marker not in cmdline:
            raise RuntimeError(
                f"WRONG server on port {port}: PID {pid} ({cmdline or '?'}) does not contain "
                f"'{expected_marker}'. Run the teardown cell, then the launch cell."
            )
    if proc is not None:
        mine = _own_pids(proc)
        if not owners <= mine:
            raise RuntimeError(
                f"port {port} is owned by "
                + ", ".join(_describe(pid) for pid in sorted(owners))
                + f", not by the server this notebook started (PID {proc.pid})"
            )
    print(f"OK: {expected_marker} (PID {min(owners)}) owns port {port}")


def gpu_report() -> None:
    """Print GPU memory/utilisation and the processes using the GPU."""
    queries = (
        ["nvidia-smi", "--query-gpu=name,memory.used,memory.total,utilization.gpu",
         "--format=csv"],
        ["nvidia-smi", "--query-compute-apps=pid,process_name,used_memory", "--format=csv"],
    )
    for argv in queries:
        try:
            result = subprocess.run(argv, capture_output=True, text=True, timeout=30)
        except FileNotFoundError:
            print("no nvidia-smi (CPU runtime?): pick a GPU runtime for this notebook")
            return
        print(result.stdout.rstrip() or result.stderr.rstrip())


def health_report(port: int) -> dict:
    """GET /health on the local server, pretty-print it and return it."""
    url = f"http://127.0.0.1:{port}/health"
    try:
        with urllib.request.urlopen(url, timeout=10) as response:
            health = json.loads(response.read().decode())
    except Exception as exc:
        print(f"GET {url} failed: {type(exc).__name__}: {exc}")
        raise
    print(json.dumps(health, indent=2, sort_keys=True))
    return health


def stop_server(proc, port: int, patterns: Sequence[str] = ()) -> None:
    """Teardown: stop `proc` (if alive) and any leftover server, then confirm the port is free."""
    if proc is not None and proc.poll() is None:
        proc.terminate()
        try:
            proc.wait(timeout=10)
        except subprocess.TimeoutExpired:
            proc.kill()
            proc.wait(timeout=10)
        print(f"Stopped PID {proc.pid}")
    stop_servers(port, patterns)
    print(f"port {port} is free")


In [ ]:
import hashlib
from pathlib import Path

EXPECTED_SHA256 = '2d8f63cbfdf5e679c268dd4f4436129e8fa8f30acfd5c5f5d88eb1082c2bf2a8'
actual = hashlib.sha256(Path('/content/colab_server_utils.py').read_bytes()).hexdigest()
if actual != EXPECTED_SHA256:
    raise RuntimeError(
        f'/content/colab_server_utils.py has sha256 {actual[:12]}, expected {EXPECTED_SHA256[:12]}: '
        're-run the %%writefile cell above. To change the file, edit control/vla_bridge/colab_server_utils.py '
        'and regenerate the notebooks (scripts/build_colab_notebooks.py).'
    )
print(f'/content/colab_server_utils.py matches control/vla_bridge/colab_server_utils.py (sha256 {actual[:12]})')

## Tier C: Wall-X / WALL-OSS resident adapter (Plan 14-08, `--backend wallx`)

The installed `lerobot==0.6.1` `PolicyServer` does not dispatch `wall_x`: it is not in
`async_inference/constants.py`'s `SUPPORTED_POLICIES`. So Wall-X can't use the gRPC lane. It runs
as its own resident Flask adapter, `control/vla_bridge/wallx_server.py`, built like the
OpenVLA-OFT adapter (see README): it loads once, warms up, then serves.

The adapter loads Wall-X through **lerobot's own** `WallXPolicy` (`lerobot[wallx]==0.6.1`, a
pure-Python extra). It does **not** clone or `pip install` `github.com/X-Square-Robot/wall-x`. That
route needs a source build of flash-attn, `dmuon` from a second GitHub repo, and an nvcc
`CUDAExtension` compile (14-RESEARCH.md Pitfall 5). This path has none of those steps.

**Status:** written from installed lerobot 0.6.1 source and HF Hub metadata. These cells have
**not yet been run on Colab**.

Requirements:
- **GPU:** an A100. The pinned fine-tune and the base model it builds on are 8.4 GB of
  safetensors each, and `WallXPolicy` builds the model in fp32 before casting it to bf16.
- **RAM:** a High-RAM runtime is likely needed for the same reason.

Default checkpoint: `Zekai-Chen/wallx_pick_place_3cam_200ep@ef8ce55f9772`. It is trained on
SO-101 in degree mode, for the single task "Put the object in the bin", with cameras
`wrist`/`top`/`side`.

In [ ]:
# Wall-X only: lerobot's own `wallx` extra, same lerobot pin as the gRPC notebooks (pure Python:
# transformers>=5.4,<5.6, peft, scipy, torchdiffeq, qwen-vl-utils). No GitHub clone and no
# CUDA-operator compile. Both the install and the import check are verified by exit status,
# not by a quiet pip run.
import subprocess, sys
install = subprocess.run([sys.executable, '-m', 'pip', 'install', 'lerobot[wallx]==0.6.1', 'flask'],
                         capture_output=True, text=True)
print(install.stdout[-2000:]); print(install.stderr[-2000:])
if install.returncode != 0:
    raise RuntimeError(f'lerobot[wallx] install failed (exit {install.returncode})')
# Fresh interpreter: this kernel may still hold the pre-install transformers in memory.
check = subprocess.run([sys.executable, '-c', (
    "import importlib.metadata as m\n"
    "from lerobot.utils.import_utils import _wallx_deps_available\n"
    "from lerobot.policies.factory import get_policy_class\n"
    "assert _wallx_deps_available, 'lerobot[wallx] dependencies are missing'\n"
    "print(get_policy_class('wall_x'))\n"
    "print({p: m.version(p) for p in ('lerobot', 'torch', 'transformers', 'peft', 'qwen-vl-utils')})\n"
)], capture_output=True, text=True)
print(check.stdout); print(check.stderr[-2000:])
if check.returncode != 0:
    raise RuntimeError('lerobot[wallx] import check failed; see stderr above')

## Adapter source and launch notes

The next cell writes `control/vla_bridge/wallx_server.py` into `/content/wallx_server.py`, and the cell after it checks its sha256. The launch cell then starts it as a separate OS process.

On first start it snapshot-downloads
two pinned revisions, about 17 GB in total:
- the fine-tune, `Zekai-Chen/wallx_pick_place_3cam_200ep@ef8ce55f9772`
- the base model it builds on, `x-square-robot/wall-oss-flow@44e827683819`

`/health` only answers after the model has loaded and warmed up. Logs go to
`/content/wallx_server.log`.

To use a different SO-101 `wall_x` checkpoint, pass `--checkpoint <repo> --revision <sha>`.
Also pass `--camera-map key=source,...`, mapping each of its camera keys to `wrist`,
`overhead_left` or `overhead_right`. The server refuses to start if any camera is unmapped.

In [ ]:
%%writefile /content/wallx_server.py
"""Resident Wall-X / WALL-OSS HTTP adapter (Tier C, Plan 14-08). Run on Colab
with `lerobot[wallx]==0.6.1` installed in the same Python.

Why an HTTP adapter and not the gRPC bridge: the installed `lerobot==0.6.1`
async `PolicyServer` only dispatches `async_inference/constants.py::
SUPPORTED_POLICIES` = act, smolvla, diffusion, tdmpc, vqbet, pi0, pi05, groot
(enforced in `policy_server.py:130`). `wall_x` is not in it, even though
`lerobot/policies/wall_x/` ships a full `WallXPolicy`.

This mirrors `openvla_oft_server.py` / `fastfs_server.py`:
- the model loads ONCE
- one throwaway warmup inference runs before the socket opens
- `/health` reports ready
- a non-blocking `inference_lock` answers concurrent requests with 429
- `ValueError`/`KeyError`/`TypeError` on a request become 400, and a model
  failure (`RuntimeError`, e.g. a non-finite action) becomes 500, so the
  client holds position instead of executing a bogus action

Unlike OpenVLA-OFT it returns the `action_contract.JOINT_ORDER`-keyed absolute
joint dict directly (Wall-X SO-101 fine-tunes are joint-space), so the client
runs with `convert_fn=None` (`base_http_adapter` contract).

Model loading: lerobot's own WALL-OSS integration, NOT the X-Square-Robot/wall-x
GitHub install
-----------------------------------------------------------------------------
14-RESEARCH.md scoped the `github.com/X-Square-Robot/wall-x` source install to
the case where Wall-X is NOT reachable through `lerobot.policies`. It is:
`lerobot/policies/wall_x/` is "adapted from their open-source WallX repository"
(lerobot docs/source/walloss.mdx @ v0.6.1), and the `lerobot[wallx]` extra is
pure Python (transformers>=5.4,<5.6, peft, scipy, torchdiffeq, qwen-vl-utils).
The GitHub route would instead need a source build of flash-attn 2.8.3,
`dmuon` from a second GitHub repo, lerobot `main` with `--no-deps`, and an
nvcc `CUDAExtension` compile (wall-x `setup.py` @ ce2a1ae). That compile is
14-RESEARCH.md Pitfall 5, and this path does not have it.

Loading follows lerobot 0.6.1's own inference path (`async_inference/
policy_server.py` SendPolicyInstructions + `common/control_utils.predict_action`):
`PreTrainedConfig.from_pretrained` -> `get_policy_class("wall_x").from_pretrained`
-> `make_pre_post_processors(..., pretrained_path=...)` ->
`prepare_observation_for_inference` -> preprocessor -> policy -> postprocessor.
Two 0.6.1-specific details, both verified against installed source:
- `WallXConfig.__post_init__` raises unless `attn_implementation == "eager"`,
  and some hub checkpoints saved `"sdpa"` (e.g. Zekai-Chen/wallx_pick_place_3cam).
  The config is therefore parsed with the draccus override
  `--attn_implementation=eager`.
- `WallXPolicy.__init__` loads the base model from
  `config.pretrained_name_or_path` (`x-square-robot/wall-oss-flow`) with no
  revision argument, i.e. `main`. To honor the pinned base revision (T-14-06)
  the base is snapshot-downloaded at its sha and the override points
  `--pretrained_name_or_path` at that local directory.

Default checkpoint (HF API read 2026-10-01)
-------------------------------------------
Zekai-Chen/wallx_pick_place_3cam_200ep @ ef8ce55f9772c4c76728228324eebe1d2c373ff6
- Apache-2.0, `type: wall_x`, `prediction_mode: diffusion`,
  `attn_implementation: eager`, chunk 32, 60k steps on 200 episodes.
- Inputs: `observation.state` [6] plus `observation.images.{wrist,top,side}`
  (3x480x640). Output: `action` [6].
- Dataset Zekai-Chen/so101_pi05_pick_place_3cam (Apache-2.0, `so_follower`,
  300 episodes, 30 fps, one task "Put the object in the bin"). Its state and
  action names are exactly `shoulder_pan.pos .. gripper.pos` in JOINT_ORDER,
  in degree mode (shoulder_lift down to -103.5, wrist_roll down to -135.3,
  gripper 0..63). This matches `action_contract.ACTION_UNITS`.
- Base: x-square-robot/wall-oss-flow @ 44e827683819957d8c574e8b746a1a97e77f518a
  (Qwen2.5-VL MoE, model.safetensors 8.4 GB; the fine-tune is another 8.4 GB).
`x-square-robot/wall-oss-0.5` (the plan's suggestion) is the raw foundation
model: `Qwen2_5_VLForConditionalGeneration`, no `type: wall_x`, and no SO-101
normalizer stats, so it cannot produce an SO-101 joint action.

Camera mapping: the client sends `wrist`, `overhead_left` and `overhead_right`
(the AR0144 stereo halves). `--camera-map` assigns each checkpoint image key
to one of those, and startup refuses a checkpoint whose cameras are not all
mapped. The default map (wrist->wrist, top->overhead_left,
side->overhead_right) fits the default checkpoint. Our rig has no side camera,
so `side` gets the right stereo half: out of distribution, which is acceptable
for this phase's "one validated live action" bar.

Only the FIRST action of each 32-step chunk is returned (receding horizon,
same as openvla_oft_server.py). The control loop re-queries every tick.

Colab launch: see vla_bridge/notebooks/http_wallx.ipynb.
"""
import argparse
import base64
import math
import threading
import time
from pathlib import Path

import cv2
import numpy as np

POLICY_TYPE = "wall_x"
VERIFIED_LEROBOT_VERSION = "0.6.1"
DEFAULT_CHECKPOINT = "Zekai-Chen/wallx_pick_place_3cam_200ep"
# T-14-06: pin the reviewed HF revisions (HF API sha read 2026-10-01), never `main`.
DEFAULT_REVISION = "ef8ce55f9772c4c76728228324eebe1d2c373ff6"
DEFAULT_BASE_MODEL = "x-square-robot/wall-oss-flow"
DEFAULT_BASE_REVISION = "44e827683819957d8c574e8b746a1a97e77f518a"
DEFAULT_LOCAL_DIR = "/content/wallx_checkpoints"
DEFAULT_PORT = 3002
# Copy of action_contract.JOINT_ORDER: this file is uploaded to Colab on its own.
JOINT_ORDER = ("shoulder_pan", "shoulder_lift", "elbow_flex", "wrist_flex", "wrist_roll", "gripper")
CAMERA_SOURCES = ("wrist", "overhead_left", "overhead_right")
DEFAULT_CAMERA_MAP = {"wrist": "wrist", "top": "overhead_left", "side": "overhead_right"}
IMAGE_PREFIX = "observation.images."
STATE_KEY = "observation.state"
ACTION_KEY = "action"
MAX_IMAGE_SIDE = 1024


def _installed_lerobot_version():
    from importlib.metadata import version

    return version("lerobot")


def _download_pinned(repo_id, revision, local_dir):
    """Materialise the exact pinned HF revision as a real directory."""
    from huggingface_hub import snapshot_download

    target = Path(local_dir) / repo_id.replace("/", "--") / revision
    return snapshot_download(repo_id=repo_id, revision=revision, local_dir=str(target))


def config_overrides(base_dir, device):
    """draccus overrides applied when parsing the checkpoint's config.json."""
    return [
        "--attn_implementation=eager",
        f"--pretrained_name_or_path={base_dir}",
        f"--device={device}",
    ]


def parse_camera_map(text):
    """`"top=overhead_left,side=overhead_right,wrist=wrist"` -> dict."""
    camera_map = {}
    for item in str(text).split(","):
        if not item.strip():
            continue
        key, sep, source = (part.strip() for part in item.partition("="))
        if not sep or not key or source not in CAMERA_SOURCES:
            raise ValueError(f"camera map entry {item!r} must be <checkpoint key>=<one of {CAMERA_SOURCES}>")
        if key in camera_map:
            raise ValueError(f"camera map assigns {key!r} twice")
        camera_map[key] = source
    if not camera_map:
        raise ValueError("camera map is empty")
    return camera_map


def _load_policy(checkpoint_dir, base_dir, device):
    """lerobot 0.6.1's own Wall-X load path (see module docstring)."""
    import lerobot.policies  # noqa: F401  (registers WallXConfig with draccus)
    from lerobot.configs.policies import PreTrainedConfig
    from lerobot.policies.factory import get_policy_class, make_pre_post_processors
    from lerobot.policies.utils import prepare_observation_for_inference

    config = PreTrainedConfig.from_pretrained(checkpoint_dir, cli_overrides=config_overrides(base_dir, device))
    if config.type != POLICY_TYPE:
        raise ValueError(f"{checkpoint_dir} is a {config.type!r} checkpoint, not {POLICY_TYPE!r}")
    policy = get_policy_class(POLICY_TYPE).from_pretrained(checkpoint_dir, config=config)
    policy.to(device)
    policy.eval()
    preprocessor, postprocessor = make_pre_post_processors(
        policy.config,
        pretrained_path=checkpoint_dir,
        preprocessor_overrides={
            "device_processor": {"device": device},
            "rename_observations_processor": {"rename_map": {}},
        },
        postprocessor_overrides={"device_processor": {"device": "cpu"}},
    )
    return dict(config=policy.config, policy=policy, preprocessor=preprocessor,
                postprocessor=postprocessor, prepare_observation=prepare_observation_for_inference)


def _feature_dim(feature):
    shape = tuple(getattr(feature, "shape", ()) or ())
    return shape[0] if len(shape) == 1 else None


class WallXModel:
    def __init__(self, checkpoint=DEFAULT_CHECKPOINT, revision=DEFAULT_REVISION,
                 base_model=DEFAULT_BASE_MODEL, base_revision=DEFAULT_BASE_REVISION,
                 camera_map=None, local_dir=DEFAULT_LOCAL_DIR, device="cuda"):
        checkpoint_dir = _download_pinned(checkpoint, revision, local_dir) if revision else checkpoint
        base_dir = _download_pinned(base_model, base_revision, local_dir) if base_revision else base_model
        loaded = _load_policy(checkpoint_dir, base_dir, device)
        config = loaded["config"]
        self._validate_config(checkpoint, config)

        self.camera_map = dict(DEFAULT_CAMERA_MAP if camera_map is None else camera_map)
        image_keys = [key[len(IMAGE_PREFIX):] for key in config.image_features]
        unmapped = [key for key in image_keys if key not in self.camera_map]
        if unmapped:
            raise ValueError(
                f"{checkpoint} declares cameras {image_keys}; {unmapped} have no --camera-map entry "
                f"(map each to one of {CAMERA_SOURCES})"
            )
        self.image_keys = image_keys
        self.required_sources = tuple(dict.fromkeys(self.camera_map[key] for key in image_keys))

        self.device = device
        self.policy = loaded["policy"]
        self.preprocessor = loaded["preprocessor"]
        self.postprocessor = loaded["postprocessor"]
        self._prepare_observation = loaded["prepare_observation"]
        self.model_version = f"{checkpoint}@{revision[:12] if revision else 'unpinned'}"
        self.metadata = dict(
            checkpoint=checkpoint, revision=revision or "unpinned",
            base_model=base_model, base_revision=base_revision or "unpinned",
            policy_type=POLICY_TYPE, prediction_mode=getattr(config, "prediction_mode", None),
            chunk_size=getattr(config, "chunk_size", None), action_space="absolute_joint",
            action_units="degrees, gripper percent", joint_order=list(JOINT_ORDER),
            camera_map={key: self.camera_map[key] for key in image_keys},
            returns="first action of chunk",
        )

    @staticmethod
    def _validate_config(checkpoint, config):
        if getattr(config, "type", None) != POLICY_TYPE:
            raise ValueError(f"{checkpoint} is a {getattr(config, 'type', None)!r} checkpoint, not {POLICY_TYPE!r}")
        action_dim = _feature_dim(config.output_features.get(ACTION_KEY))
        state_dim = _feature_dim(config.input_features.get(STATE_KEY))
        if action_dim != len(JOINT_ORDER) or state_dim != len(JOINT_ORDER):
            raise ValueError(
                f"{checkpoint} has state dim {state_dim} / action dim {action_dim}; an SO-101 "
                f"joint-space checkpoint needs {len(JOINT_ORDER)} / {len(JOINT_ORDER)}"
            )

    def __call__(self, images, joint_state, instruction):
        """`images`: source name -> HxWx3 uint8 RGB. `joint_state`: JOINT_ORDER
        dict (degrees, gripper percent). Returns the absolute JOINT_ORDER dict."""
        import torch

        observation = {IMAGE_PREFIX + key: images[self.camera_map[key]] for key in self.image_keys}
        observation[STATE_KEY] = np.asarray([joint_state[j] for j in JOINT_ORDER], dtype=np.float32)
        with torch.inference_mode():
            batch = self._prepare_observation(observation, torch.device(self.device), instruction, "")
            batch = self.preprocessor(batch)
            chunk = self.policy.predict_action_chunk(batch)
            first = self.postprocessor(chunk[:, 0, :])
        values = np.asarray(first.detach().cpu().float().numpy() if hasattr(first, "detach") else first,
                            dtype=np.float64).reshape(-1)
        if values.shape != (len(JOINT_ORDER),):
            raise RuntimeError(f"model returned action of shape {values.shape}, expected ({len(JOINT_ORDER)},)")
        if not np.all(np.isfinite(values)):
            raise RuntimeError(f"model returned a non-finite action {values.tolist()}")
        return {joint: float(v) for joint, v in zip(JOINT_ORDER, values)}


def _decode_rgb(b64_png):
    """base64 PNG (OpenCV BGR-encoded by the client) -> HxWx3 uint8 RGB."""
    if not isinstance(b64_png, str):
        raise TypeError("image must be a base64 string")
    raw = base64.b64decode(b64_png, validate=True)
    frame = cv2.imdecode(np.frombuffer(raw, dtype=np.uint8), cv2.IMREAD_COLOR)
    if frame is None:
        raise ValueError("invalid PNG")
    if max(frame.shape[:2]) > MAX_IMAGE_SIDE:
        raise ValueError(f"image dimensions must be <= {MAX_IMAGE_SIDE}")
    return cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)


def _parse_joint_state(value):
    if not isinstance(value, dict):
        raise TypeError("joint_state must be an object keyed by joint name")
    state = {}
    for joint in JOINT_ORDER:
        v = value[joint]
        if isinstance(v, bool) or not isinstance(v, (int, float)) or not math.isfinite(v):
            raise ValueError(f"joint_state[{joint!r}] must be a finite number")
        state[joint] = float(v)
    return state


def create_app(infer):
    from flask import Flask, jsonify, request
    app = Flask(__name__)
    app.config['MAX_CONTENT_LENGTH'] = 20 * 1024 * 1024
    inference_lock = threading.Lock()

    @app.get('/health')
    def health():
        return jsonify(status='ready', protocol_version=1, model=infer.metadata)

    @app.post('/action')
    def action():
        if not inference_lock.acquire(blocking=False):
            return jsonify(error='wall-x service busy'), 429
        try:
            body = request.get_json(silent=True)
            if not isinstance(body, dict):
                raise ValueError('expected a JSON object body')
            images_in = body['images']
            if not isinstance(images_in, dict):
                raise TypeError('images must be an object')
            images = {key: _decode_rgb(images_in[key]) for key in infer.required_sources}
            instruction = body['instruction']
            if not isinstance(instruction, str) or not instruction.strip():
                raise ValueError('instruction must be a non-empty string')
            joint_state = _parse_joint_state(body['joint_state'])
        except (ValueError, KeyError, TypeError) as exc:
            inference_lock.release()
            return jsonify(error=str(exc)), 400
        try:
            started = time.perf_counter()
            result = infer(images, joint_state, instruction)
            elapsed = (time.perf_counter() - started) * 1000
            return jsonify(action={j: float(result[j]) for j in JOINT_ORDER},
                           model_version=infer.model_version, inference_ms=elapsed)
        except RuntimeError as exc:
            return jsonify(error=f'wall-x inference failed: {exc}'), 500
        finally:
            inference_lock.release()
    return app


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--checkpoint', default=DEFAULT_CHECKPOINT)
    parser.add_argument('--revision', default=DEFAULT_REVISION,
                        help="pinned HF commit sha; pass '' only to deliberately load the unpinned repo id")
    parser.add_argument('--base-model', default=DEFAULT_BASE_MODEL)
    parser.add_argument('--base-revision', default=DEFAULT_BASE_REVISION,
                        help="pinned sha of the base model WallXPolicy loads; '' loads its main branch")
    parser.add_argument('--camera-map', default=None,
                        help="checkpoint camera key -> client source, e.g. 'top=overhead_left,wrist=wrist'")
    parser.add_argument('--local-dir', default=DEFAULT_LOCAL_DIR)
    parser.add_argument('--device', default='cuda')
    parser.add_argument('--port', type=int, default=DEFAULT_PORT)
    args = parser.parse_args()
    installed = _installed_lerobot_version()
    if installed != VERIFIED_LEROBOT_VERSION:
        raise SystemExit(
            f"wallx_server.py was verified against lerobot=={VERIFIED_LEROBOT_VERSION} (WallXConfig's eager-only "
            f"check, processor step names, load path); found {installed}. Install lerobot[wallx]=="
            f"{VERIFIED_LEROBOT_VERSION} or re-verify this file against the new source."
        )
    camera_map = parse_camera_map(args.camera_map) if args.camera_map else None
    model = WallXModel(checkpoint=args.checkpoint, revision=args.revision or None,
                       base_model=args.base_model, base_revision=args.base_revision or None,
                       camera_map=camera_map, local_dir=args.local_dir, device=args.device)
    # CUDA kernels / lazy init happen before the socket opens and readiness is reported.
    warmup = np.zeros((480, 640, 3), dtype=np.uint8)
    model({source: warmup for source in CAMERA_SOURCES}, {j: 0.0 for j in JOINT_ORDER}, 'warm up')
    print('Wall-X model loaded and warmed; ready', flush=True)
    create_app(model).run(host='0.0.0.0', port=args.port, threaded=True, use_reloader=False)


if __name__ == '__main__':
    main()


In [ ]:
import hashlib
from pathlib import Path

EXPECTED_SHA256 = '9302581708bfda8bc5cf26dd4abf983a4a1340177d626b20db9a12bdd0e619c4'
actual = hashlib.sha256(Path('/content/wallx_server.py').read_bytes()).hexdigest()
if actual != EXPECTED_SHA256:
    raise RuntimeError(
        f'/content/wallx_server.py has sha256 {actual[:12]}, expected {EXPECTED_SHA256[:12]}: '
        're-run the %%writefile cell above. To change the file, edit control/vla_bridge/wallx_server.py '
        'and regenerate the notebooks (scripts/build_colab_notebooks.py).'
    )
print(f'/content/wallx_server.py matches control/vla_bridge/wallx_server.py (sha256 {actual[:12]})')

## Launch

The launch cell first stops any earlier server: every process whose command line contains one of `wallx_server.py`, and whatever listens on port 3002.
It then starts this notebook's server as a separate OS process (unbuffered, log at `/content/wallx_server.log`).
It only succeeds once **its own** still-running process owns port 3002.
A stale server answering on the port is an error, never a successful launch (in the 2026-10-18 live session a stale stock server made a diffusion launch look successful).

In [ ]:
import sys

sys.path.insert(0, '/content')
from colab_server_utils import start_server, stop_servers, wait_until_ready

PORT = 3002
WALLX_LOG = '/content/wallx_server.log'
STOP_PATTERNS = ['wallx_server.py']

stop_servers(PORT, STOP_PATTERNS)
wallx_server_process = start_server(
    [sys.executable, '-u', '/content/wallx_server.py', '--port', '3002'],
    WALLX_LOG,
    env_overrides={'TOKENIZERS_PARALLELISM': 'false'},
)
health = wait_until_ready(
    wallx_server_process, PORT, WALLX_LOG,
    timeout_s=3600, health_url='http://127.0.0.1:3002/health',
)
print('Wall-X adapter ready')

## Open an HTTP tunnel to port 3002

In [ ]:
from pyngrok import ngrok

wallx_tunnel = ngrok.connect(3002, "http")  # plain JSON-over-HTTP
print(f"Wall-X endpoint: {wallx_tunnel.public_url}")
SMOKE = ".venv/bin/python -m vla_bridge.backend_smoke --backend wallx --server-address https://<wallx_tunnel> --instruction 'Put the object in the bin'"
EPISODE = '.venv/bin/python run_vla_episode.py --backend wallx --server-address https://WALLX_TUNNEL.ngrok-free.app --instruction "Put the object in the bin" --out outputs/wallx_UNIQUE_NAME --max-steps 20'
for placeholder in ('https://<wallx_tunnel>', 'https://WALLX_TUNNEL.ngrok-free.app'):
    SMOKE = SMOKE.replace(placeholder, wallx_tunnel.public_url)
    EPISODE = EPISODE.replace(placeholder, wallx_tunnel.public_url)
print('Local smoke (run in control/):')
print(SMOKE)
print('Local episode (run in control/):')
print(EPISODE)

## Run locally

Then, locally in `control/` (pass the base URL; the client appends `/action`):

```bash
.venv/bin/python run_vla_episode.py --backend wallx \
  --server-address https://WALLX_TUNNEL.ngrok-free.app \
  --instruction "Put the object in the bin" \
  --out outputs/wallx_UNIQUE_NAME --max-steps 20
```

`--instruction` matches the default checkpoint's only training task. Each step's
`model_version` should read `Zekai-Chen/wallx_pick_place_3cam_200ep@ef8ce55f9772`. A step that
logs `http-error-holding-position` means the adapter returned an error. Check
`/content/wallx_server.log`: a 500 `wall-x inference failed` means the model produced a
non-finite action, and a 400 means a bad request.

Smoke run first (synthetic hardware, the arm cannot move; the tunnel cell prints it with the URL filled in):

```bash
.venv/bin/python -m vla_bridge.backend_smoke --backend wallx --server-address https://<wallx_tunnel> --instruction 'Put the object in the bin'
```

## Diagnostics

Run any of these after the launch cell, also from a restarted kernel: each cell re-declares
what it needs.
- Log: the tail of `/content/wallx_server.log`, then every line matching `ERROR`, `Error`, `Traceback`,
  `Exception`, `OutOfMemory`, `out of memory`, `Killed` or `beginAllocateToPool`.
- Processes: every known server process on this runtime (stale servers from other notebooks
  show up here); fails unless `wallx_server.py` owns port 3002.
- Package versions, GPU memory, and `/health`.

In [ ]:
import sys

sys.path.insert(0, '/content')
from colab_server_utils import log_report

log_report('/content/wallx_server.log')

In [ ]:
import sys

sys.path.insert(0, '/content')
from colab_server_utils import process_report

process_report(3002, 'wallx_server.py', globals().get('wallx_server_process'))

In [ ]:
import subprocess
import sys

probe = (
    "import importlib.metadata as m\n"
    "print({p: m.version(p) for p in ('lerobot', 'torch', 'transformers', 'peft', 'qwen-vl-utils')})\n"
)
result = subprocess.run([sys.executable, '-c', probe], capture_output=True, text=True)
print(result.stdout)
print(result.stderr[-2000:])

In [ ]:
import sys

sys.path.insert(0, '/content')
from colab_server_utils import gpu_report

gpu_report()

In [ ]:
import sys

sys.path.insert(0, '/content')
from colab_server_utils import health_report

health_report(3002)

## Teardown (every session, no exceptions)

An unattended open tunnel into a Colab-hosted server is an
elevation-of-privilege risk (T-11-08 in this plan's threat register) —
stop both processes at the end of every session, not just when finished
for the day: the cell below disconnects this notebook's tunnel, stops the
ngrok agent, stops the server and confirms port 3002 is free. Run it only
after the local episode or smoke run has finished.

In [ ]:
# Wall-X teardown -- every session, after the local run has finished.
import sys

sys.path.insert(0, '/content')
from colab_server_utils import stop_server
from pyngrok import ngrok

if 'wallx_tunnel' in globals():
    try:
        ngrok.disconnect(wallx_tunnel.public_url)
        print(f'Disconnected {wallx_tunnel.public_url}')
    except Exception as exc:
        print(f'Tunnel already closed ({type(exc).__name__}: {exc})')
ngrok.kill()
print('ngrok agent stopped.')
stop_server(globals().get('wallx_server_process'), 3002, ['wallx_server.py'])